In [3]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

from ase.io import read, write
from ase.mep import NEB
from ase.optimize import MDMin, BFGS
from ase.constraints import FixAtoms
from mace.calculators import MACECalculator

# =========================
# CONFIG
# =========================
model_path = "/home/mehuldarak/athena/mace_fps_training/mace_fps_split17_compiled.model"
structure_path = "/home/mehuldarak/athena/Case3VASP/Case3.cif"

device = "cuda" if torch.cuda.is_available() else "cpu"

n_images = 9
freeze_framework = True   # recommended for LLZO
traj_stage1 = "neb_stage1.traj"
traj_stage2 = "neb_stage2.traj"
traj_ci     = "neb_ci.traj"

# =========================
# CALCULATOR
# =========================
def get_calc():
    return MACECalculator(
        model_paths=[model_path],
        device=device
    )

# =========================
# BUILD VACANCY NEB ENDPOINTS
# =========================
atoms = read(structure_path)

symbols = atoms.get_chemical_symbols()
li = [i for i, s in enumerate(symbols) if s == "Li"]

if len(li) < 2:
    raise RuntimeError("Need at least 2 Li atoms")

# pick vacancy + hopping Li
i_vac = li[1]
vac_pos = atoms.positions[i_vac].copy()

atoms.pop(i_vac)

i0 = li[0]
if i_vac < i0:
    i0 -= 1

initial = atoms.copy()
final   = atoms.copy()
final.positions[i0] = vac_pos

# =========================
# RELAX ENDPOINTS (IMPORTANT)
# =========================
for name, a in [("initial", initial), ("final", final)]:
    a.calc = get_calc()
    opt = BFGS(a, logfile=f"{name}_relax.log")
    opt.run(fmax=0.02)
    write(f"{name}_relaxed.traj", a)

# reload relaxed (clean state)
initial = read("initial_relaxed.traj")
final   = read("final_relaxed.traj")

# =========================
# CREATE IMAGES
# =========================
images = [initial.copy()]
for _ in range(n_images - 2):
    images.append(initial.copy())
images.append(final.copy())

# =========================
# OPTIONAL: FREEZE FRAMEWORK
# =========================
if freeze_framework:
    symbols = images[0].get_chemical_symbols()
    mask = [s != "Li" for s in symbols]
    constraint = FixAtoms(mask=mask)
else:
    constraint = None

# attach calc + constraints
for img in images:
    img.calc = get_calc()
    img.set_constraint(None)
    if constraint is not None:
        img.set_constraint(constraint)

# =========================
# NEB SETUP
# =========================
neb = NEB(images, k=0.2, climb=False)

# IMPORTANT: good initial path
neb.interpolate(method="idpp", apply_constraint=False)

# =========================
# STAGE 1: STABILIZE (MDMin)
# =========================
print("\n--- Stage 1: MDMin ---")
opt = MDMin(
    neb,
    dt=0.05,
    trajectory=traj_stage1,
    logfile="neb_stage1.log"
)
opt.run(fmax=0.08)

# =========================
# STAGE 2: REFINE (BFGS)
# =========================
print("\n--- Stage 2: BFGS ---")
opt = BFGS(
    neb,
    trajectory=traj_stage2,
    logfile="neb_stage2.log"
)
opt.run(fmax=0.03)

# =========================
# STAGE 3: CLIMBING IMAGE
# =========================
print("\n--- Stage 3: Climbing Image ---")
neb = NEB(images, k=0.2, climb=True)

opt = BFGS(
    neb,
    trajectory=traj_ci,
    logfile="neb_ci.log"
)
opt.run(fmax=0.03)

# =========================
# RESULTS
# =========================
energies = np.array([img.get_potential_energy() for img in images])
E0 = energies[0]
rel = energies - E0

barrier = rel.max()
imax = rel.argmax()

print("\n===== NEB RESULTS =====")
for i, e in enumerate(rel):
    print(f"Image {i}: {e:.6f} eV")

print(f"\nBarrier: {barrier:.6f} eV")
print(f"Saddle image index: {imax}")

# =========================
# SAVE SADDLE STRUCTURE
# =========================
write("saddle.cif", images[imax])

# =========================
# PLOT
# =========================
plt.figure(figsize=(6,4))
plt.plot(rel, marker='o')
plt.xlabel("Image")
plt.ylabel("Energy (eV)")
plt.title("Li Migration NEB (MACE)")
plt.grid()
plt.tight_layout()
plt.savefig("neb_profile.png", dpi=200)
plt.show()

Using head Default out of ['Default']
No dtype selected, switching to float32 to match model dtype.
Using head Default out of ['Default']
No dtype selected, switching to float32 to match model dtype.
Using head Default out of ['Default']
No dtype selected, switching to float32 to match model dtype.
Using head Default out of ['Default']
No dtype selected, switching to float32 to match model dtype.
Using head Default out of ['Default']
No dtype selected, switching to float32 to match model dtype.
Using head Default out of ['Default']
No dtype selected, switching to float32 to match model dtype.
Using head Default out of ['Default']
No dtype selected, switching to float32 to match model dtype.
Using head Default out of ['Default']
No dtype selected, switching to float32 to match model dtype.
Using head Default out of ['Default']
No dtype selected, switching to float32 to match model dtype.
Using head Default out of ['Default']
No dtype selected, switching to float32 to match model dtype.


KeyboardInterrupt: 